In [2]:

# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================
!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [3]:

# ==============================
# 2. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import ADASYN
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=a1b15917873954f486195977e683b05e604e5d0b7b21630067cac0b05dc04bb5
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [4]:

# ==============================
# 3. LOAD DATASET
# ==============================
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [5]:


# ==============================
# 4. PREPROCESSING
# ==============================
df = df.drop('duration', axis=1)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

df['y'] = df['y'].map({'no': 0, 'yes': 1})

df = pd.get_dummies(df, drop_first=True)

In [6]:


# ==============================
# 5. SPLIT DATA
# ==============================
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [8]:

# ADASYN
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)


In [9]:

# ==============================
# 6. METRIC FUNCTIONS
# ==============================
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

In [10]:

# ==============================
# 8. CROSS VALIDATION (MAIN)
# ==============================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Random Forest

In [11]:

#Import Random Forest
from sklearn.ensemble import RandomForestClassifier

In [12]:
#Hyperparameter Tuning
# ==============================
# RANDOM FOREST TUNING
# ==============================
params_rf = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rscv_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    params_rf,
    n_iter=5,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_rf.fit(X_train_ad, y_train_ad)

best_rf = rscv_rf.best_estimator_

In [13]:
#Test Performance
print("\n🌲 Random Forest (Tuned)")

y_pred_rf = best_rf.predict(X_test_scaled)
y_prob_rf = best_rf.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_rf, y_prob_rf)

print("Best Params (RF):", rscv_rf.best_params_)


🌲 Random Forest (Tuned)
Accuracy    : 0.8853
Precision   : 0.4878
Recall      : 0.3653
Specificity : 0.9513
F1 Score    : 0.4177
ROC-AUC     : 0.7750
MCC         : 0.3601
G-Mean      : 0.5895
Kappa       : 0.3556
Best Params (RF): {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}


In [14]:
#Cross Validation (Random Forest)
results_rf = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (optional for RF, but keeping consistent)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_rf.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_rf.predict(X_test_f)
    y_prob = best_rf.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_rf.append({
        "Fold": fold+1,
        "Classifier": "Random Forest",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [15]:

df_rf  = pd.DataFrame(results_rf)

#final_df = pd.concat([df_cat, df_lr, df_dt, df_rf])

final_df = pd.concat([df_rf])

In [16]:

print("\n===== Random Forest MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== Random Forest MODELS =====
 Fold    Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Random Forest  0.885166   0.487465 0.377155     0.949658 0.425273 0.598472 0.050342 0.773790 0.366320 0.362635           0.663407          10.551377
    2 Random Forest  0.882010   0.466463 0.329741     0.952120 0.386364 0.560316 0.047880 0.751658 0.329174 0.323216           0.640931          10.435219
    3 Random Forest  0.873027   0.420912 0.338362     0.940903 0.375149 0.564239 0.059097 0.775027 0.307667 0.305412           0.639632          10.566456
    4 Random Forest  0.883710   0.479893 0.385776     0.946922 0.427718 0.604400 0.053078 0.758586 0.366534 0.363847           0.666349          10.709796
    5 Random Forest  0.888565   0.507163 0.381466     0.952941 0.435424 0.602921 0.047059 0.773565 0.379659 0.374975           0.667203          10.652748
    6 Random Forest  0.884681   0.48

# **Decision Tree**

In [17]:

from sklearn.tree import DecisionTreeClassifier

In [18]:

# ==============================
# DECISION TREE TUNING
# ==============================
params_dt = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

rscv_dt = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params_dt,
    n_iter=5,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_dt.fit(X_train_ad, y_train_ad)

best_dt = rscv_dt.best_estimator_

In [19]:
print("\n🌳 Decision Tree (Tuned)")

y_pred_dt = best_dt.predict(X_test_scaled)
y_prob_dt = best_dt.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_dt, y_prob_dt)

print("Best Params (DT):", rscv_dt.best_params_)


🌳 Decision Tree (Tuned)
Accuracy    : 0.8613
Precision   : 0.3665
Recall      : 0.3179
Specificity : 0.9302
F1 Score    : 0.3405
ROC-AUC     : 0.6640
MCC         : 0.2642
G-Mean      : 0.5438
Kappa       : 0.2634
Best Params (DT): {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': None, 'criterion': 'entropy'}


In [20]:
#Cross Validation (Decision Tree)

results_dt = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_dt.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_dt.predict(X_test_f)
    y_prob = best_dt.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_dt.append({
        "Fold": fold+1,
        "Classifier": "Decision Tree",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })


In [21]:
#All Model
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
df_dt  = pd.DataFrame(results_dt)

final_df = pd.concat([ df_dt])

In [23]:
#Print Comparison
print("\n===== Decision Tree MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== Decision Tree MODELS =====
 Fold    Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1 Decision Tree  0.865501   0.371429 0.280172     0.939808 0.319410 0.513136 0.060192 0.649502 0.249425 0.246408           0.609990           0.676957
    2 Decision Tree  0.857004   0.329700 0.260776     0.932695 0.291215 0.493178 0.067305 0.644566 0.214711 0.212899           0.596735           0.995204
    3 Decision Tree  0.856276   0.345411 0.308190     0.925855 0.325740 0.534171 0.074145 0.649091 0.246097 0.245597           0.617022           0.714634
    4 Decision Tree  0.862102   0.372549 0.327586     0.929959 0.348624 0.551944 0.070041 0.670962 0.272572 0.271868           0.628773           0.699924
    5 Decision Tree  0.870842   0.410995 0.338362     0.938440 0.371158 0.563500 0.061560 0.666516 0.301703 0.299941           0.638401           0.688050
    6 Decision Tree  0.857247   0.35

# **K Nearest Neighbour **

In [24]:

from sklearn.neighbors import KNeighborsClassifier

In [25]:
# ==============================
# KNN TUNING
# ==============================
params_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

rscv_knn = RandomizedSearchCV(
    KNeighborsClassifier(),
    params_knn,
    n_iter=5,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_knn.fit(X_train_ad, y_train_ad)

best_knn = rscv_knn.best_estimator_

In [26]:
print("\n KNN (Tuned)")

y_pred_knn = best_knn.predict(X_test_scaled)
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_knn, y_prob_knn)

print("Best Params (KNN):", rscv_knn.best_params_)


 KNN (Tuned)
Accuracy    : 0.8018
Precision   : 0.2839
Recall      : 0.4989
Specificity : 0.8402
F1 Score    : 0.3619
ROC-AUC     : 0.6987
MCC         : 0.2691
G-Mean      : 0.6475
Kappa       : 0.2549
Best Params (KNN): {'weights': 'distance', 'n_neighbors': 3, 'metric': 'euclidean'}


In [27]:
results_knn = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (VERY IMPORTANT for KNN ⚠️)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_knn.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_knn.predict(X_test_f)
    y_prob = best_knn.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_knn.append({
        "Fold": fold+1,
        "Classifier": "KNN",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [28]:
df_knn = pd.DataFrame(results_knn)

final_df = pd.concat([df_knn])


In [29]:
print("\n===== KNN MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))


===== KNN MODELS =====
 Fold Classifier  Accuracy  Precision   Recall  Specificity       F1       GM      FPR      AUC      MCC    Kappa  Balanced Accuracy  Training Time (s)
    1        KNN  0.805778   0.289474 0.497845     0.844870 0.366086 0.648548 0.155130 0.695096 0.274157 0.260774           0.671357           0.005051
    2        KNN  0.799223   0.271698 0.465517     0.841587 0.343129 0.625918 0.158413 0.684161 0.246022 0.234180           0.653552           0.004435
    3        KNN  0.798009   0.260417 0.431034     0.844596 0.324675 0.603366 0.155404 0.674687 0.223750 0.214332           0.637815           0.006722
    4        KNN  0.805293   0.282776 0.474138     0.847332 0.354267 0.633839 0.152668 0.694630 0.259666 0.248162           0.660735           0.004311
    5        KNN  0.806992   0.292346 0.502155     0.845691 0.369548 0.651666 0.154309 0.706172 0.278395 0.264868           0.673923           0.005383
    6        KNN  0.806992   0.290771 0.495690     0.846512 0.36

# **FriedmanTest**

In [31]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Friedman Test
from scipy.stats import friedmanchisquare

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("bank-additional-full.csv", sep=';')

print("Dataset Shape :", df.shape)

# =========================================================
# DATA PREPROCESSING
# =========================================================

# Convert categorical columns to numeric
label_encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = label_encoder.fit_transform(df[col])

# Separate Features and Target
X = df.drop("y", axis=1)
y = df["y"]

# Handle Missing Values
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# =========================================================
# DEFINE MACHINE LEARNING MODELS
# =========================================================

models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

# =========================================================
# CROSS VALIDATION
# =========================================================

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

print("\n========== MODEL ACCURACY ==========\n")

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=kfold,
        scoring='accuracy'
    )

    results[name] = scores

    print(name)
    print("Accuracy Scores :", scores)
    print("Average Accuracy :", np.mean(scores))
    print("-----------------------------------")

# =========================================================
# FRIEDMAN TEST
# =========================================================

statistic, p_value = friedmanchisquare(
    results["Random Forest"],
    results["Decision Tree"],
    results["KNN"],
    results["Logistic Regression"]
)

# =========================================================
# FINAL RESULT
# =========================================================

print("\n===================================")
print("         FRIEDMAN TEST")
print("===================================\n")

print("Friedman Statistic :", statistic)
print("P-value            :", p_value)

alpha = 0.05

print("\n========== INTERPRETATION ==========\n")

if p_value < alpha:
    print("Reject Null Hypothesis (H0)")
    print("There is a significant difference")
    print("between the machine learning models.")
else:
    print("Fail to Reject Null Hypothesis (H0)")
    print("No significant difference")
    print("between the machine learning models.")

# =========================================================
# END
# =========================================================

Dataset Shape : (41188, 21)

========== MODEL ACCURACY ==========

Random Forest
Accuracy Scores : [0.91187181 0.9152707  0.91429959 0.91453199 0.91732427]
Average Accuracy : 0.9146596711885978
-----------------------------------
Decision Tree
Accuracy Scores : [0.88370964 0.88868657 0.88892935 0.88673061 0.89304358]
Average Accuracy : 0.888219950817908
-----------------------------------
KNN
Accuracy Scores : [0.90094683 0.90058267 0.9066521  0.90676217 0.90712638]
Average Accuracy : 0.9044140298264545
-----------------------------------
Logistic Regression
Accuracy Scores : [0.90677349 0.90907987 0.91211459 0.91283234 0.91198252]
Average Accuracy : 0.9105565626331581
-----------------------------------

         FRIEDMAN TEST

Friedman Statistic : 15.0
P-value            : 0.0018166489665723214

========== INTERPRETATION ==========

Reject Null Hypothesis (H0)
There is a significant difference
between the machine learning models.
